In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# 1.Load Data

df = pd.read_csv('/content/Car_Price_Prediction.csv') # Replace with your actual CSV file path
print("Original Data Shape:", df.shape)
print(df.head())

Original Data Shape: (1000, 8)
    Make    Model  Year  Engine Size  Mileage Fuel Type Transmission  \
0  Honda  Model B  2015          3.9    74176    Petrol       Manual   
1   Ford  Model C  2014          1.7    94799  Electric    Automatic   
2    BMW  Model B  2006          4.1    98385  Electric       Manual   
3  Honda  Model B  2015          2.6    88919  Electric    Automatic   
4  Honda  Model C  2004          3.4   138482    Petrol    Automatic   

          Price  
0  30246.207931  
1  22785.747684  
2  25760.290347  
3  25638.003491  
4  21021.386657  


In [3]:
# 2.Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())


Data Types:
 Make             object
Model            object
Year              int64
Engine Size     float64
Mileage           int64
Fuel Type        object
Transmission     object
Price           float64
dtype: object

Missing Values:
 Make            0
Model           0
Year            0
Engine Size     0
Mileage         0
Fuel Type       0
Transmission    0
Price           0
dtype: int64


In [4]:
# 3.Handle Missing Values

# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Numerical: Replace with mean
num_imputer = SimpleImputer(strategy='mean')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Categorical: Replace with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [5]:
# 4.Encode Categorical Variables

# Identify columns to one-hot encode based on expected features for X and y
ohe_cols = ['Make', 'Model', 'Fuel Type', 'Transmission']

# Perform One-hot Encoding for the specified categorical columns
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

In [6]:
# 5.Outlier Detection and Treatment (using IQR)

for col in num_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df[col] = np.where(df[col] < lower, lower, df[col])
df[col] = np.where(df[col] > upper, upper, df[col])

In [7]:
# 6.Feature Scaling

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [8]:
from sklearn.feature_selection import SelectKBest, f_regression
target_col = 'Price'  # Replace with your actual target column name

if target_col in df.columns:
    X = df.drop(target_col, axis=1)
    y = df[target_col]

    bestfeatures = SelectKBest(score_func=f_regression, k=10)
    fit = bestfeatures.fit(X, y)

    selected_features = X.columns[fit.get_support()]
    df = df[selected_features.to_list() + [target_col]]

    print("\nSelected Features:", selected_features.to_list())

corr = df.corr()
corr.sort_values(by='Price', ascending=False)


Selected Features: ['Year', 'Engine Size', 'Mileage', 'Make_Ford', 'Make_Honda', 'Make_Toyota', 'Model_Model C', 'Model_Model D', 'Fuel Type_Petrol', 'Transmission_Manual']


,Year,Engine Size,Mileage,Make_Ford,Make_Honda,Make_Toyota,Model_Model C,Model_Model D,Fuel Type_Petrol,Transmission_Manual,Price
Price,0.610079,0.383705,-0.557073,0.019624,0.054394,-0.020922,0.010732,0.013125,-0.011254,-0.027074,1.000000
Year,1.000000,-0.012190,0.016376,-0.011354,0.090540,-0.042703,-0.001986,0.023786,-0.047579,-0.007501,0.610079
Engine Size,-0.012190,1.000000,-0.014815,0.031074,0.027792,0.004304,0.042708,0.029069,-0.020838,0.030035,0.383705
Make_Honda,0.090540,0.027792,0.000593,-0.267723,1.000000,-0.238298,0.002549,-0.006347,-0.056197,-0.025994,0.054394
Make_Ford,-0.011354,0.031074,-0.016672,1.000000,-0.267723,-0.258414,0.028918,-0.038083,0.033206,-0.042996,0.019624
Model_Model D,0.023786,0.029069,0.060616,-0.038083,-0.006347,0.091314,-0.251518,1.000000,0.009580,0.046942,0.013125
Model_Model C,-0.001986,0.042708,0.006705,0.028918,0.002549,-0.027540,1.000000,-0.251518,-0.009765,-0.028518,0.010732
Fuel Type_Petrol,-0.047579,-0.020838,-0.048279,0.033206,-0.056197,-0.026689,-0.009765,0.009580,1.000000,0.020656,-0.011254
Make_Toyota,-0.042703,0.004304,-0.005723,-0.258414,-0.238298,1.000000,-0.027540,0.091314,-0.026689,0.058710,-0.020922
Transmission_Manual,-0.007501,0.030035,0.058043,-0.042996,-0.025994,0.058710,-0.028518,0.046942,0.020656,1.000000,-0.027074


In [11]:
import numpy as np
import sklearn.preprocessing as preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = np.asarray(df[['Year', 'Engine Size', 'Mileage', 'Make_Honda', 'Make_Toyota', 'Model_Model D', 'Fuel Type_Petrol', 'Transmission_Manual']])
y = np.asarray(df['Price'])

# Scale the features using preprocessing.StandardScaler()
X = preprocessing.StandardScaler().fit(X).transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size = 0.3, random_state = 100)

print ('Train set:', X_train.shape, y_train.shape)
print ('Test set:', X_test.shape, y_test.shape)

Train set: (700, 8) (700,)
Test set: (300, 8) (300,)


In [12]:
#Regression Models
#1.Linear Regression

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.8338916718246662


In [13]:
#2.Decision Tree

from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.6369795971477586


In [14]:
#3.Random Forest
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.7927299782420132


In [15]:
#4.K-Nearest Neighbors

from sklearn.neighbors import KNeighborsRegressor
model = KNeighborsRegressor(n_neighbors=10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.706002912995698


In [16]:
#5.Support Vector Regression

from sklearn.svm import SVR
model = SVR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.7994924460288032


In [17]:
#6.Gradient Boosting

from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.8125083085812592


In [18]:
#7.Ridge Regression

from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.833820045967772
